# LeetCode 1396 — Design Underground System bằng Python

Link đề: https://leetcode.com/problems/design-underground-system/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Design

Skeleton LeetCode:

```python
class UndergroundSystem:

    def __init__(self):
        ...

    def checkIn(self, id: int, stationName: str, t: int) -> None:
        ...

    def checkOut(self, id: int, stationName: str, t: int) -> None:
        ...

    def getAverageTime(self, startStation: str, endStation: str) -> float:
        ...
```

> Đây là bài **thiết kế lớp**. Notebook tự dựng một **bộ chạy lệnh giả lập** kiểu LeetCode (nhận danh sách tên lệnh + tham số, trả về danh sách kết quả) để kiểm thử. Bộ chạy này là bản giả lập, không phải trình chấm thật.

## 1. Mục tiêu học

1. Biết **thiết kế lớp có trạng thái**: chọn dữ liệu cần lưu sao cho mỗi thao tác chạy `O(1)`.
2. Dùng thành thạo `dict` với **khoá tuple** `(ga_đi, ga_đến)` và giá trị `[tổng, số_chuyến]`.
3. So sánh 4 thiết kế: lưu mọi chuyến (baseline), lưu danh sách thời gian theo cặp ga, **tổng + đếm** (tối ưu), và **trung bình chạy** (running mean).
4. Tự viết **bộ chạy lệnh giả lập** kiểu LeetCode và so sánh số thực với **sai số `1e-5`**.
5. Tránh lỗi: ghép chuỗi làm khoá bị trùng, chia nguyên `//`, lưu check-in theo ga thay vì theo id, gộp hai chiều A→B và B→A.

## 2. Đề bài & ví dụ

Hệ thống tàu điện ngầm ghi lại thời gian di chuyển của khách để tính **thời gian trung bình** giữa hai ga. Cài đặt lớp `UndergroundSystem`:

- `checkIn(id, stationName, t)`: khách có thẻ `id` vào ga `stationName` lúc `t`. Mỗi khách chỉ check-in ở một nơi tại một thời điểm.
- `checkOut(id, stationName, t)`: khách `id` ra ở ga `stationName` lúc `t`.
- `getAverageTime(startStation, endStation)`: trả về thời gian trung bình của **mọi chuyến đã hoàn thành** đi **thẳng** từ `startStation` tới `endStation` (check-in ở `startStation` rồi check-out ở `endStation`). Thời gian A→B **có thể khác** B→A. Đảm bảo đã có ít nhất một chuyến như vậy khi hàm được gọi.

Mọi lời gọi đều **nhất quán**: check-in lúc `t1` rồi check-out lúc `t2` thì `t1 < t2`; các sự kiện theo thứ tự thời gian.

**Ví dụ 1** (rút gọn): ba khách 45, 32, 27 check-in; 45 đi Leyton→Waterloo mất 12, 27 đi Leyton→Waterloo mất 10, 32 đi Paradise→Cambridge mất 14. `getAverageTime("Paradise","Cambridge") = 14.0`, `getAverageTime("Leyton","Waterloo") = (10+12)/2 = 11.0`. Sau đó khách 10 đi Leyton→Waterloo mất 14 ⇒ trung bình `(10+12+14)/3 = 12.0`.

**Ví dụ 2:** ba chuyến Leyton→Paradise lần lượt 5, 6, 9 ⇒ trung bình sau mỗi chuyến: `5.0`, `5.5`, `6.66667`.

**Ràng buộc:** `1 <= id, t <= 10^6`; tên ga dài 1–10 ký tự (chữ hoa, chữ thường, chữ số); tối đa `2·10^4` lời gọi; đáp án sai số trong `10^-5` được chấp nhận.

In [1]:
import math
import random
import time
from collections import defaultdict
from typing import Dict, List, Tuple

EX1_OPS = ["UndergroundSystem", "checkIn", "checkIn", "checkIn", "checkOut", "checkOut", "checkOut",
           "getAverageTime", "getAverageTime", "checkIn", "getAverageTime", "checkOut", "getAverageTime"]
EX1_ARGS = [[], [45, "Leyton", 3], [32, "Paradise", 8], [27, "Leyton", 10], [45, "Waterloo", 15],
            [27, "Waterloo", 20], [32, "Cambridge", 22], ["Paradise", "Cambridge"], ["Leyton", "Waterloo"],
            [10, "Leyton", 24], ["Leyton", "Waterloo"], [10, "Waterloo", 38], ["Leyton", "Waterloo"]]
EX1_OUT = [None, None, None, None, None, None, None, 14.0, 11.0, None, 11.0, None, 12.0]

EX2_OPS = ["UndergroundSystem", "checkIn", "checkOut", "getAverageTime", "checkIn", "checkOut",
           "getAverageTime", "checkIn", "checkOut", "getAverageTime"]
EX2_ARGS = [[], [10, "Leyton", 3], [10, "Paradise", 8], ["Leyton", "Paradise"], [5, "Leyton", 10],
            [5, "Paradise", 16], ["Leyton", "Paradise"], [2, "Leyton", 21], [2, "Paradise", 30],
            ["Leyton", "Paradise"]]
EX2_OUT = [None, None, None, 5.0, None, None, 5.5, None, None, 6.66667]

for op, a in zip(EX2_OPS, EX2_ARGS):
    print(f"{op}({', '.join(map(repr, a))})")

UndergroundSystem()
checkIn(10, 'Leyton', 3)
checkOut(10, 'Paradise', 8)
getAverageTime('Leyton', 'Paradise')
checkIn(5, 'Leyton', 10)
checkOut(5, 'Paradise', 16)
getAverageTime('Leyton', 'Paradise')
checkIn(2, 'Leyton', 21)
checkOut(2, 'Paradise', 30)
getAverageTime('Leyton', 'Paradise')


## 3. Kiến thức nền

### 3.1 Lớp và trạng thái (`self`)

Một đối tượng lưu dữ liệu trong các **thuộc tính** `self.x`. Mọi phương thức của cùng đối tượng đều đọc/ghi chung các thuộc tính đó — đó là "bộ nhớ" giữa các lời gọi.

In [2]:
class Counter:
    def __init__(self):
        self.count = 0          # trạng thái khởi tạo

    def inc(self):
        self.count += 1

    def get(self):
        return self.count

c = Counter(); c.inc(); c.inc()
print(c.get())

2


### 3.2 `dict` với khoá là `tuple`

Tuple là bất biến (hashable) nên làm khoá `dict` được. `(start, end)` là khoá tự nhiên cho "cặp ga có thứ tự": `("A", "B") != ("B", "A")`.

In [3]:
stats = {}
stats[("Leyton", "Waterloo")] = [22, 2]
stats[("Waterloo", "Leyton")] = [30, 1]
print(stats[("Leyton", "Waterloo")], stats[("Waterloo", "Leyton")])
print(("A", "B") == ("B", "A"))

[22, 2] [30, 1]
False


### 3.3 `dict.pop(key)` — lấy ra và xoá

Khi khách check-out, ta lấy thông tin check-in **và xoá luôn** để `dict` không phình mãi. `pop` trả về giá trị và xoá khoá trong `O(1)`.

In [4]:
check_in = {45: ("Leyton", 3), 32: ("Paradise", 8)}
station, t1 = check_in.pop(45)
print(station, t1, "| còn lại:", check_in)

Leyton 3 | còn lại: {32: ('Paradise', 8)}


### 3.4 `defaultdict` để khỏi kiểm tra khoá tồn tại

`defaultdict(lambda: [0, 0])` tự tạo `[0, 0]` khi truy cập khoá mới.

In [5]:
stats = defaultdict(lambda: [0, 0])
for start, end, d in [("A", "B", 5), ("A", "B", 7), ("B", "A", 4)]:
    s = stats[(start, end)]
    s[0] += d
    s[1] += 1
print(dict(stats))

{('A', 'B'): [12, 2], ('B', 'A'): [4, 1]}


### 3.5 Chia số thực `/` và chia nguyên `//`

Trong Python 3, `/` luôn trả `float`; `//` là chia lấy phần nguyên. Trung bình phải dùng `/`.

In [6]:
print(11 / 2, 11 // 2, 20 / 3)

5.5 5 6.666666666666667


### 3.6 So sánh số thực với sai số

`0.1 + 0.2 != 0.3` vì sai số dấu phẩy động. Đề chấp nhận sai số `1e-5` ⇒ so bằng `abs(a - b) <= 1e-5` hoặc `math.isclose(a, b, abs_tol=1e-5)`.

In [7]:
print(0.1 + 0.2 == 0.3, abs((0.1 + 0.2) - 0.3) <= 1e-5)
print(20 / 3, "vs 6.66667:", 20 / 3 == 6.66667, math.isclose(20 / 3, 6.66667, abs_tol=1e-5))

False True
6.666666666666667 vs 6.66667: False True


### 3.7 Bộ chạy lệnh giả lập kiểu LeetCode

LeetCode cho hai mảng: tên lệnh và tham số. Lệnh đầu là tạo đối tượng; các lệnh sau gọi phương thức cùng tên bằng `getattr(obj, name)(*args)`. Ta viết hàm `run` để dùng lại cho mọi thiết kế, và `same` để so kết quả có sai số.

In [8]:
def run(cls, ops, args):
    """Bộ chạy GIẢ LẬP: tạo đối tượng từ lệnh đầu, gọi lần lượt các lệnh sau."""
    obj = cls(*args[0])
    out = [None]
    for op, a in zip(ops[1:], args[1:]):
        out.append(getattr(obj, op)(*a))
    return out

def same(got, want, eps=1e-5):
    if len(got) != len(want):
        return False
    for g, w in zip(got, want):
        if w is None:
            if g is not None:
                return False
        elif g is None or abs(g - w) > eps:
            return False
    return True

print(run(Counter, ["Counter", "inc", "inc", "get"], [[], [], [], []]))

[None, None, None, 2]


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1**, ghi lại hai thứ: ai đang ở trong hệ thống (đã check-in, chưa check-out) và thống kê theo cặp ga.

| Lệnh | Đang trong hệ thống | Thống kê (tổng, số chuyến) |
|---|---|---|
| checkIn(45, Leyton, 3) | 45: Leyton@3 | — |
| checkIn(32, Paradise, 8) | 45, 32: Paradise@8 | — |
| checkIn(27, Leyton, 10) | 45, 32, 27: Leyton@10 | — |
| checkOut(45, Waterloo, 15) | 32, 27 | Leyton→Waterloo: (12, 1) |
| checkOut(27, Waterloo, 20) | 32 | Leyton→Waterloo: (22, 2) |
| checkOut(32, Cambridge, 22) | — | + Paradise→Cambridge: (14, 1) |
| getAverageTime(Paradise, Cambridge) | | 14/1 = **14.0** |
| getAverageTime(Leyton, Waterloo) | | 22/2 = **11.0** |
| checkIn(10, Leyton, 24) | 10: Leyton@24 | |
| getAverageTime(Leyton, Waterloo) | | vẫn **11.0** (khách 10 chưa xong) |
| checkOut(10, Waterloo, 38) | — | Leyton→Waterloo: (36, 3) |
| getAverageTime(Leyton, Waterloo) | | 36/3 = **12.0** |

Nhận xét: để tính trung bình ta **không cần nhớ từng chuyến**, chỉ cần **tổng thời gian** và **số chuyến** cho mỗi cặp ga.

In [9]:
trips = [("Leyton", "Waterloo", 15 - 3), ("Leyton", "Waterloo", 20 - 10), ("Paradise", "Cambridge", 22 - 8)]
total = defaultdict(int); cnt = defaultdict(int)
for a, b, d in trips:
    total[(a, b)] += d; cnt[(a, b)] += 1
for key in total:
    print(key, "tổng", total[key], "số chuyến", cnt[key], "-> TB", total[key] / cnt[key])

('Leyton', 'Waterloo') tổng 22 số chuyến 2 -> TB 11.0
('Paradise', 'Cambridge') tổng 14 số chuyến 1 -> TB 14.0


## 5. Cách 1 — Brute Force / baseline: lưu mọi chuyến, quét khi hỏi

### Ý tưởng
- `checkIn`: lưu `id → (ga, t)`.
- `checkOut`: tạo bản ghi chuyến `(ga_đi, ga_đến, thời_gian)` và **thêm vào danh sách** chung.
- `getAverageTime`: **duyệt toàn bộ danh sách**, lọc các chuyến đúng cặp ga, tính trung bình.

In [10]:
class UndergroundBrute:
    def __init__(self):
        self.active = {}          # id -> (ga vào, thời điểm vào)
        self.trips = []           # mọi chuyến đã xong: (start, end, duration)

    def checkIn(self, id, stationName, t):
        self.active[id] = (stationName, t)

    def checkOut(self, id, stationName, t):
        start, t1 = self.active.pop(id)
        self.trips.append((start, stationName, t - t1))

    def getAverageTime(self, startStation, endStation):
        ds = [d for s, e, d in self.trips if s == startStation and e == endStation]
        return sum(ds) / len(ds)

for ops, args, want in [(EX1_OPS, EX1_ARGS, EX1_OUT), (EX2_OPS, EX2_ARGS, EX2_OUT)]:
    got = run(UndergroundBrute, ops, args)
    print(got, "| khớp:", same(got, want))

[None, None, None, None, None, None, None, 14.0, 11.0, None, 11.0, None, 12.0] | khớp: True
[None, None, None, 5.0, None, None, 5.5, None, None, 6.666666666666667] | khớp: True


### Độ phức tạp và nút thắt

Gọi `T` = số chuyến đã hoàn thành.

- `checkIn`, `checkOut`: `O(1)`.
- `getAverageTime`: `O(T)` — duyệt **mọi** chuyến của **mọi** cặp ga.
- Bộ nhớ `O(T + P)` với `P` = số khách đang trong hệ thống.

Với `2·10^4` lời gọi, nếu một nửa là check-out và nửa kia là truy vấn ⇒ ~`10^4 × 10^4 = 10^8` phép so sánh: quá chậm. Nút thắt: tính lại từ đầu thứ mà ta có thể **cộng dồn dần**.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** trung bình = `tổng / số_lượng`. Cả tổng lẫn số lượng đều **cộng dồn được**: mỗi chuyến mới chỉ cần `tổng += d`, `số += 1`. Vậy ta duy trì **hai bảng băm**:

1. `check_in: id → (ga_vào, t_vào)` — khách đang trong hệ thống.
2. `stats: (ga_đi, ga_đến) → [tổng_thời_gian, số_chuyến]` — thống kê theo cặp ga **có thứ tự**.

**Bất biến:** sau mỗi lệnh, `stats[(A, B)]` = (tổng, số) của đúng các chuyến A→B **đã check-out**; `check_in` chứa đúng các khách đang ở trong hệ thống.

**Quy tắc "nếu… thì…":**
- Nếu `checkIn` ⇒ ghi `check_in[id]`.
- Nếu `checkOut` ⇒ `pop` thông tin vào, cộng `t − t_vào` vào `stats[(ga_vào, ga_ra)]`.
- Nếu `getAverageTime` ⇒ trả `tổng / số` (chia **số thực**).

## 7. Các cách tiếp theo

### Cách 2 — Danh sách thời gian theo cặp ga

Cải tiến nhỏ: thay vì một danh sách chung, gom chuyến theo khoá `(start, end)`. Truy vấn chỉ duyệt các chuyến **đúng cặp** — nhanh hơn baseline nhưng vẫn `O(số chuyến của cặp đó)`.

In [11]:
class UndergroundLists:
    def __init__(self):
        self.active = {}
        self.durations = defaultdict(list)     # (start, end) -> [d1, d2, ...]

    def checkIn(self, id, stationName, t):
        self.active[id] = (stationName, t)

    def checkOut(self, id, stationName, t):
        start, t1 = self.active.pop(id)
        self.durations[(start, stationName)].append(t - t1)

    def getAverageTime(self, startStation, endStation):
        ds = self.durations[(startStation, endStation)]
        return sum(ds) / len(ds)

for ops, args, want in [(EX1_OPS, EX1_ARGS, EX1_OUT), (EX2_OPS, EX2_ARGS, EX2_OUT)]:
    print(same(run(UndergroundLists, ops, args), want))

True
True


Ưu điểm: giữ được dữ liệu chi tiết (có thể tính trung vị, min, max sau này). Nhược: truy vấn vẫn tuyến tính theo số chuyến của cặp; bộ nhớ `O(T)`.

### Cách 3 — Tổng + đếm (tối ưu, `O(1)` mỗi thao tác)

**Từng đoạn:**
1. `__init__`: hai `dict` rỗng.
2. `checkIn`: `self.check_in[id] = (stationName, t)`.
3. `checkOut`: `start, t1 = self.check_in.pop(id)`; lấy `s = self.stats.setdefault((start, stationName), [0, 0])`; `s[0] += t - t1`; `s[1] += 1`.
4. `getAverageTime`: `total, cnt = self.stats[(start, end)]`; `return total / cnt`.

In [12]:
class UndergroundSumCount:
    def __init__(self):
        self.check_in = {}        # id -> (ga vào, t vào)
        self.stats = {}           # (ga đi, ga đến) -> [tổng, số chuyến]

    def checkIn(self, id, stationName, t):
        self.check_in[id] = (stationName, t)

    def checkOut(self, id, stationName, t):
        start, t1 = self.check_in.pop(id)
        s = self.stats.setdefault((start, stationName), [0, 0])
        s[0] += t - t1
        s[1] += 1

    def getAverageTime(self, startStation, endStation):
        total, cnt = self.stats[(startStation, endStation)]
        return total / cnt

for ops, args, want in [(EX1_OPS, EX1_ARGS, EX1_OUT), (EX2_OPS, EX2_ARGS, EX2_OUT)]:
    got = run(UndergroundSumCount, ops, args)
    print(got, "| khớp:", same(got, want))

[None, None, None, None, None, None, None, 14.0, 11.0, None, 11.0, None, 12.0] | khớp: True
[None, None, None, 5.0, None, None, 5.5, None, None, 6.666666666666667] | khớp: True


Mọi thao tác `O(1)` trung bình (băm chuỗi tên ga dài ≤ 10 coi như hằng số). Bộ nhớ `O(P + S²)` với `S` là số ga xuất hiện (thực tế chỉ các cặp có chuyến). Bản nộp LeetCode của cách này ở mục 9.

### Cách 4 — Trung bình chạy (running mean)

Thay vì lưu tổng, lưu thẳng **trung bình hiện tại** `avg` và số chuyến `cnt`. Khi có chuyến mới `d`: `cnt += 1`, `avg += (d − avg) / cnt`. Truy vấn trả `avg` không cần chia.

Lợi ích: tổng không bao giờ phình to (hữu ích ở ngôn ngữ có tràn số, hoặc khi dữ liệu rất lớn). Bất lợi: mỗi lần cập nhật có **sai số làm tròn** nhỏ; ở đây sai số ≪ `1e-5` nên vẫn được chấp nhận. Python có số nguyên vô hạn nên Cách 3 vẫn là lựa chọn chính xác nhất.

In [13]:
class UndergroundRunningMean:
    def __init__(self):
        self.check_in = {}
        self.mean = {}            # (start, end) -> [trung bình, số chuyến]

    def checkIn(self, id, stationName, t):
        self.check_in[id] = (stationName, t)

    def checkOut(self, id, stationName, t):
        start, t1 = self.check_in.pop(id)
        m = self.mean.setdefault((start, stationName), [0.0, 0])
        m[1] += 1
        m[0] += ((t - t1) - m[0]) / m[1]

    def getAverageTime(self, startStation, endStation):
        return self.mean[(startStation, endStation)][0]

for ops, args, want in [(EX1_OPS, EX1_ARGS, EX1_OUT), (EX2_OPS, EX2_ARGS, EX2_OUT)]:
    got = run(UndergroundRunningMean, ops, args)
    print(got, "| khớp:", same(got, want))

[None, None, None, None, None, None, None, 14.0, 11.0, None, 11.0, None, 12.0] | khớp: True
[None, None, None, 5.0, None, None, 5.5, None, None, 6.666666666666667] | khớp: True


Kiểm tra công thức running mean: sau các giá trị `5, 6, 9` ta có `5 → 5 + (6−5)/2 = 5.5 → 5.5 + (9−5.5)/3 = 6.6667` ✔.

In [14]:
avg, cnt = 0.0, 0
for d in [5, 6, 9]:
    cnt += 1
    avg += (d - avg) / cnt
    print(f"thêm {d}: cnt={cnt}, avg={avg:.6f}")

thêm 5: cnt=1, avg=5.000000
thêm 6: cnt=2, avg=5.500000
thêm 9: cnt=3, avg=6.666667


## 8. TRACE từng bước

Hàm `trace_system` chạy chuỗi lệnh trên thiết kế Cách 3 và in **hai bảng băm** sau mỗi lệnh.

In [15]:
def trace_system(ops, args):
    obj = UndergroundSumCount()
    for op, a in zip(ops[1:], args[1:]):
        res = getattr(obj, op)(*a)
        call = f"{op}({', '.join(map(repr, a))})"
        extra = f" -> {res:.5f}" if res is not None else ""
        print(f"{call:38s}{extra}")
        print(f"    check_in = {obj.check_in}")
        print(f"    stats    = {obj.stats}")

trace_system(EX1_OPS, EX1_ARGS)

checkIn(45, 'Leyton', 3)              
    check_in = {45: ('Leyton', 3)}
    stats    = {}
checkIn(32, 'Paradise', 8)            
    check_in = {45: ('Leyton', 3), 32: ('Paradise', 8)}
    stats    = {}
checkIn(27, 'Leyton', 10)             
    check_in = {45: ('Leyton', 3), 32: ('Paradise', 8), 27: ('Leyton', 10)}
    stats    = {}
checkOut(45, 'Waterloo', 15)          
    check_in = {32: ('Paradise', 8), 27: ('Leyton', 10)}
    stats    = {('Leyton', 'Waterloo'): [12, 1]}
checkOut(27, 'Waterloo', 20)          
    check_in = {32: ('Paradise', 8)}
    stats    = {('Leyton', 'Waterloo'): [22, 2]}
checkOut(32, 'Cambridge', 22)         
    check_in = {}
    stats    = {('Leyton', 'Waterloo'): [22, 2], ('Paradise', 'Cambridge'): [14, 1]}
getAverageTime('Paradise', 'Cambridge') -> 14.00000
    check_in = {}
    stats    = {('Leyton', 'Waterloo'): [22, 2], ('Paradise', 'Cambridge'): [14, 1]}
getAverageTime('Leyton', 'Waterloo')   -> 11.00000
    check_in = {}
    stats    = {('Leyton',

Quan sát:
- Sau `checkOut(45, ...)`, khoá `45` biến mất khỏi `check_in` (đã `pop`).
- `stats[('Leyton','Waterloo')]` đi từ `[12, 1]` → `[22, 2]` → `[36, 3]`.
- Lời gọi `getAverageTime` lúc khách 10 đang đi **không** bị ảnh hưởng: chỉ chuyến đã check-out mới được tính.

Trace Ví dụ 2 gọn hơn — chỉ in kết quả truy vấn và thống kê:

In [16]:
def trace_queries(ops, args):
    obj = UndergroundSumCount()
    for op, a in zip(ops[1:], args[1:]):
        res = getattr(obj, op)(*a)
        if op == "getAverageTime":
            total, cnt = obj.stats[tuple(a)]
            print(f"{a[0]}->{a[1]}: tổng={total}, số chuyến={cnt}, TB={res:.5f}")

trace_queries(EX2_OPS, EX2_ARGS)

Leyton->Paradise: tổng=5, số chuyến=1, TB=5.00000
Leyton->Paradise: tổng=11, số chuyến=2, TB=5.50000
Leyton->Paradise: tổng=20, số chuyến=3, TB=6.66667


## 9. Code nộp LeetCode

Đúng tên lớp `UndergroundSystem` và các phương thức `checkIn`, `checkOut`, `getAverageTime`.

In [17]:
from typing import Dict, Tuple, List

class UndergroundSystem:

    def __init__(self):
        self.check_in: Dict[int, Tuple[str, int]] = {}          # id -> (ga vào, t vào)
        self.stats: Dict[Tuple[str, str], List[int]] = {}       # (đi, đến) -> [tổng, số]

    def checkIn(self, id: int, stationName: str, t: int) -> None:
        self.check_in[id] = (stationName, t)

    def checkOut(self, id: int, stationName: str, t: int) -> None:
        start, t1 = self.check_in.pop(id)
        s = self.stats.setdefault((start, stationName), [0, 0])
        s[0] += t - t1
        s[1] += 1

    def getAverageTime(self, startStation: str, endStation: str) -> float:
        total, cnt = self.stats[(startStation, endStation)]
        return total / cnt


# Your UndergroundSystem object will be instantiated and called as such:
# obj = UndergroundSystem()
# obj.checkIn(id,stationName,t)
# obj.checkOut(id,stationName,t)
# param_3 = obj.getAverageTime(startStation,endStation)

assert same(run(UndergroundSystem, EX1_OPS, EX1_ARGS), EX1_OUT)
assert same(run(UndergroundSystem, EX2_OPS, EX2_ARGS), EX2_OUT)
print("UndergroundSystem đúng với 2 ví dụ của đề (qua bộ chạy giả lập)")

UndergroundSystem đúng với 2 ví dụ của đề (qua bộ chạy giả lập)


## 10. Vì sao thuật toán đúng

**Bất biến 1 (check-in):** sau mỗi lệnh, `check_in` chứa đúng những khách đã check-in mà chưa check-out, kèm ga và thời điểm vào. `checkIn` thêm khách; `checkOut` `pop` khách. Đề đảm bảo khách chỉ ở một nơi một lúc và các lời gọi nhất quán, nên `pop(id)` luôn tìm thấy và không bao giờ ghi đè một chuyến đang dở.

**Bất biến 2 (thống kê):** `stats[(A, B)] = [Σ d, số chuyến]` trên mọi chuyến **đã hoàn thành** đi thẳng từ A tới B. Một chuyến "đi thẳng" A→B = một lần check-in tại A của khách `id` ghép với **lần check-out kế tiếp** của chính `id` tại B — đúng là cặp mà `pop` ghép lại. Mỗi chuyến được cộng đúng một lần (lúc check-out).

**Truy vấn:** trung bình theo định nghĩa = `Σ d / số chuyến` = `stats[(A,B)][0] / stats[(A,B)][1]`. Đề đảm bảo có ít nhất một chuyến ⇒ mẫu số > 0.

**Chiều đi:** khoá là tuple **có thứ tự**, nên A→B và B→A là hai khoá khác nhau — đúng yêu cầu đề.

## 11. Phân tích Time / Space

Ký hiệu: `Q` = tổng số lời gọi (≤ 2·10⁴), `T` = số chuyến đã xong, `P` = số khách đang trong hệ thống, `K` = số cặp ga có chuyến.

| Thiết kế | checkIn | checkOut | getAverageTime | Bộ nhớ |
|---|---|---|---|---|
| 1. Lưu mọi chuyến | `O(1)` | `O(1)` | `O(T)` | `O(T + P)` |
| 2. Danh sách theo cặp | `O(1)` | `O(1)` | `O(T_cặp)` | `O(T + P)` |
| 3. Tổng + đếm | `O(1)` | `O(1)` | `O(1)` | `O(K + P)` |
| 4. Trung bình chạy | `O(1)` | `O(1)` | `O(1)` | `O(K + P)` |

`O(1)` ở đây là **trung bình** của bảng băm (trường hợp xấu nhất lý thuyết `O(K)` khi va chạm băm, gần như không xảy ra). Băm chuỗi tốn `O(độ dài)` ≤ 10 ký tự ⇒ hằng số.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
DESIGNS = {
    "brute": UndergroundBrute,
    "lists": UndergroundLists,
    "sumcount": UndergroundSumCount,
    "running": UndergroundRunningMean,
    "submit": UndergroundSystem,
}
cases = [
    (EX1_OPS, EX1_ARGS, EX1_OUT),
    (EX2_OPS, EX2_ARGS, EX2_OUT),
    # hai chiều khác nhau: A->B và B->A
    (["UndergroundSystem", "checkIn", "checkOut", "checkIn", "checkOut", "getAverageTime", "getAverageTime"],
     [[], [1, "A", 1], [1, "B", 11], [2, "B", 20], [2, "A", 23], ["A", "B"], ["B", "A"]],
     [None, None, None, None, None, 10.0, 3.0]),
    # cùng một khách đi nhiều chuyến liên tiếp
    (["UndergroundSystem", "checkIn", "checkOut", "checkIn", "checkOut", "getAverageTime"],
     [[], [7, "X", 1], [7, "Y", 4], [7, "X", 10], [7, "Y", 20], ["X", "Y"]],
     [None, None, None, None, None, 6.5]),
    # đi rồi về đúng ga xuất phát (start == end)
    (["UndergroundSystem", "checkIn", "checkOut", "getAverageTime"],
     [[], [3, "Loop", 5], [3, "Loop", 9], ["Loop", "Loop"]],
     [None, None, None, 4.0]),
    # tên ga có thể "dính" nhau khi ghép chuỗi
    (["UndergroundSystem", "checkIn", "checkOut", "checkIn", "checkOut", "getAverageTime", "getAverageTime"],
     [[], [1, "A", 0], [1, "BC", 10], [2, "AB", 0], [2, "C", 30], ["A", "BC"], ["AB", "C"]],
     [None, None, None, None, None, 10.0, 30.0]),
    # giá trị lớn nhất
    (["UndergroundSystem", "checkIn", "checkOut", "getAverageTime"],
     [[], [10**6, "S1", 1], [10**6, "S2", 10**6], ["S1", "S2"]],
     [None, None, None, 999999.0]),
]
for name, cls in DESIGNS.items():
    for ops, args, want in cases:
        got = run(cls, ops, args)
        assert same(got, want), (name, got, want)
print("Mọi thiết kế đúng với", len(cases), "chuỗi lệnh")

Mọi thiết kế đúng với 7 chuỗi lệnh


### 12.2 Đối chiếu ngẫu nhiên (seed cố định)

Sinh chuỗi lệnh **nhất quán** theo đúng ràng buộc: thời gian tăng dần, khách chỉ check-out khi đang ở trong hệ thống, chỉ hỏi cặp ga đã có chuyến. Mọi thiết kế phải cho kết quả giống baseline trong sai số `1e-5`.

In [19]:
def random_ops(rng, n_calls, n_ids=6, stations=("A", "B", "C", "D", "AB", "BC")):
    ops, args = ["UndergroundSystem"], [[]]
    t = 0
    inside = {}
    done = []
    for _ in range(n_calls):
        t += rng.randint(1, 5)
        choice = rng.random()
        free = [i for i in range(1, n_ids + 1) if i not in inside]
        if choice < 0.4 and free:
            i = rng.choice(free); st = rng.choice(stations)
            inside[i] = st
            ops.append("checkIn"); args.append([i, st, t])
        elif choice < 0.8 and inside:
            i = rng.choice(sorted(inside)); st = rng.choice(stations)
            done.append((inside.pop(i), st))
            ops.append("checkOut"); args.append([i, st, t])
        elif done:
            a, b = rng.choice(done)
            ops.append("getAverageTime"); args.append([a, b])
    return ops, args

rng = random.Random(1396)
n_q = 0
for _ in range(400):
    ops, args = random_ops(rng, rng.randint(1, 60))
    ref = run(UndergroundBrute, ops, args)
    n_q += sum(op == "getAverageTime" for op in ops)
    for name, cls in DESIGNS.items():
        assert same(run(cls, ops, args), ref), name
print("400 chuỗi lệnh ngẫu nhiên khớp baseline; tổng số truy vấn đã so:", n_q)

400 chuỗi lệnh ngẫu nhiên khớp baseline; tổng số truy vấn đã so: 2985


### 12.3 Hiệu năng ở giới hạn `2·10^4` lời gọi

Kịch bản xấu cho baseline: 10⁴ chuyến cùng mọi cặp ga, sau đó 10⁴ truy vấn. (Baseline chỉ chạy với 2.000 truy vấn cho nhanh.)

In [20]:
def heavy(n_trips, n_queries):
    ops, args = ["UndergroundSystem"], [[]]
    t = 1
    for i in range(n_trips):
        ops.append("checkIn"); args.append([i, f"S{i % 7}", t]); t += 1
        ops.append("checkOut"); args.append([i, f"E{i % 5}", t]); t += 1
    for q in range(n_queries):
        ops.append("getAverageTime"); args.append([f"S{q % 7}", f"E{q % 5}"])
    return ops, args

ops_big, args_big = heavy(5000, 10000)
for name in ["lists", "sumcount", "running", "submit"]:
    t0 = time.perf_counter(); run(DESIGNS[name], ops_big, args_big)
    print(f"{name:9s}: {time.perf_counter() - t0:.3f}s")
ops_b, args_b = heavy(5000, 2000)
t0 = time.perf_counter(); run(UndergroundBrute, ops_b, args_b)
print(f"brute    : {time.perf_counter() - t0:.3f}s (chỉ 2000 truy vấn)")
assert same(run(UndergroundSystem, ops_big, args_big), run(UndergroundLists, ops_big, args_big))

lists    : 0.013s
sumcount : 0.004s
running  : 0.004s
submit   : 0.004s


brute    : 0.337s (chỉ 2000 truy vấn)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Ghép chuỗi làm khoá mà không có ký tự ngăn cách

`key = start + end` ⇒ `"A" + "BC"` và `"AB" + "C"` cùng là `"ABC"` — hai tuyến khác nhau bị trộn chung!

In [21]:
class WrongConcatKey(UndergroundSumCount):
    def checkOut(self, id, stationName, t):
        start, t1 = self.check_in.pop(id)
        s = self.stats.setdefault(start + stationName, [0, 0])     # SAI: khoá bị trùng
        s[0] += t - t1; s[1] += 1

    def getAverageTime(self, startStation, endStation):
        total, cnt = self.stats[startStation + endStation]
        return total / cnt

ops, args, want = cases[5]
print("sai :", run(WrongConcatKey, ops, args))
print("đúng:", run(UndergroundSystem, ops, args))

sai : [None, None, None, None, None, 20.0, 20.0]
đúng: [None, None, None, None, None, 10.0, 30.0]


Cả hai truy vấn đều ra `20.0` = trung bình của 10 và 30. **Sửa:** dùng tuple `(start, end)`, hoặc chèn ký tự không thể xuất hiện trong tên ga (`start + "->" + end`; tên ga chỉ có chữ và số).

### Lỗi 2 — Chia nguyên `//`

In [22]:
class WrongIntDiv(UndergroundSumCount):
    def getAverageTime(self, startStation, endStation):
        total, cnt = self.stats[(startStation, endStation)]
        return total // cnt                                        # SAI: chia nguyên

print("sai :", run(WrongIntDiv, EX2_OPS, EX2_ARGS))
print("đúng:", run(UndergroundSystem, EX2_OPS, EX2_ARGS))
print("khớp đề?", same(run(WrongIntDiv, EX2_OPS, EX2_ARGS), EX2_OUT))

sai : [None, None, None, 5, None, None, 5, None, None, 6]
đúng: [None, None, None, 5.0, None, None, 5.5, None, None, 6.666666666666667]
khớp đề? False


`5.5` thành `5`, `6.66667` thành `6`. **Sửa:** dùng `/`.

### Lỗi 3 — Lưu check-in theo **ga** thay vì theo **id khách**

Ý định "ga này có khách vào lúc t" — nhưng hai khách cùng vào một ga thì người sau **ghi đè** người trước (Ví dụ 1: khách 45 và 27 cùng vào Leyton).

In [23]:
class WrongKeyByStation:
    def __init__(self):
        self.last_in = {}        # SAI: ga -> thời điểm vào gần nhất
        self.who = {}            # id -> ga vào
        self.stats = {}

    def checkIn(self, id, stationName, t):
        self.last_in[stationName] = t
        self.who[id] = stationName

    def checkOut(self, id, stationName, t):
        start = self.who.pop(id)
        t1 = self.last_in[start]                       # lấy nhầm thời điểm của khách khác
        s = self.stats.setdefault((start, stationName), [0, 0])
        s[0] += t - t1; s[1] += 1

    def getAverageTime(self, startStation, endStation):
        total, cnt = self.stats[(startStation, endStation)]
        return total / cnt

print("sai :", run(WrongKeyByStation, EX1_OPS, EX1_ARGS))
print("đúng:", run(UndergroundSystem, EX1_OPS, EX1_ARGS))

sai : [None, None, None, None, None, None, None, 14.0, 7.5, None, 7.5, None, 9.666666666666666]
đúng: [None, None, None, None, None, None, None, 14.0, 11.0, None, 11.0, None, 12.0]


Khách 45 vào lúc 3 nhưng bị tính như vào lúc 10 ⇒ chuyến 12 phút thành 5 phút. **Sửa:** khoá theo `id` — mỗi khách một bản ghi riêng.

### Lỗi 4 — Gộp hai chiều A→B và B→A

Dùng `frozenset({start, end})` hoặc `tuple(sorted(...))` làm khoá "cho gọn" — vi phạm đề (hai chiều có thể khác nhau).

In [24]:
class WrongUndirected(UndergroundSumCount):
    def checkOut(self, id, stationName, t):
        start, t1 = self.check_in.pop(id)
        key = tuple(sorted((start, stationName)))      # SAI: mất chiều đi
        s = self.stats.setdefault(key, [0, 0])
        s[0] += t - t1; s[1] += 1

    def getAverageTime(self, startStation, endStation):
        total, cnt = self.stats[tuple(sorted((startStation, endStation)))]
        return total / cnt

ops, args, want = cases[2]
print("sai :", run(WrongUndirected, ops, args))
print("đúng:", run(UndergroundSystem, ops, args), "| mong đợi", want)

sai : [None, None, None, None, None, 6.5, 6.5]
đúng: [None, None, None, None, None, 10.0, 3.0] | mong đợi [None, None, None, None, None, 10.0, 3.0]


**Sửa:** tuple **có thứ tự** `(start, end)`.

### Lỗi 5 — So kết quả số thực bằng `==`

Khi tự kiểm thử, viết `got == [.., 6.66667]` sẽ báo sai dù thuật toán đúng, vì `20/3 = 6.666666…`.

In [25]:
got = run(UndergroundSystem, EX2_OPS, EX2_ARGS)
print("so bằng == :", got == EX2_OUT)
print("so có sai số:", same(got, EX2_OUT))

so bằng == : False
so có sai số: True


**Sửa:** so với sai số `1e-5` như đề cho phép (`abs(a − b) <= 1e-5` hoặc `math.isclose(..., abs_tol=1e-5)`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài **"Design … System"** ⇒ tự hỏi: mỗi phương thức cần **đọc** gì? Lưu sẵn đúng thứ đó để đọc `O(1)`.
- "Trung bình/tổng/đếm theo nhóm" ⇒ lưu **tổng + đếm** theo khoá nhóm, không lưu từng phần tử.
- "Ghép sự kiện mở/đóng của cùng một thực thể" (check-in/out, start/stop, login/logout) ⇒ `dict` theo **id thực thể**, `pop` khi đóng.
- Khoá ghép nhiều trường ⇒ **tuple**, đừng ghép chuỗi trần.
- Luôn tự viết bộ chạy lệnh `run(cls, ops, args)` để test nhanh nhiều thiết kế.

## 15. Biến thể của bài

1. **Thêm `getMinTime` / `getMaxTime`** cho mỗi cặp ga ⇒ lưu thêm `min`, `max` cộng dồn (Bài 4 mục 17).
2. **Trung bình mọi chuyến xuất phát từ một ga** (`getAverageFrom(start)`) ⇒ thêm `dict` thứ ba theo `start` (Bài 5).
3. **Trung vị thời gian** ⇒ không còn cộng dồn được; cần hai heap cho mỗi cặp ga (như LeetCode 295 *Find Median from Data Stream*).
4. **Chỉ tính các chuyến trong khoảng thời gian `[L, R]`** ⇒ lưu danh sách đã sắp + tổng tiền tố, tìm nhị phân.
5. **Khách huỷ chuyến** (check-in rồi không check-out) ⇒ `pop` khỏi `check_in` mà không cập nhật thống kê.
6. Họ hàng thiết kế: LeetCode 1352 *Product of the Last K Numbers*, 359 *Logger Rate Limiter*, 981 *Time Based Key-Value Store*.

## 16. So sánh các cách

| Thiết kế | Ý tưởng | Time (truy vấn) | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Lưu mọi chuyến | danh sách chung, lọc khi hỏi | `O(T)` | `O(T + P)` | bản tham chiếu, cần dữ liệu thô |
| 2. Danh sách theo cặp | `dict` cặp → list thời gian | `O(T_cặp)` | `O(T + P)` | cần thêm thống kê khác (trung vị, phân phối) |
| 3. Tổng + đếm | `dict` cặp → `[tổng, số]` | `O(1)` | `O(K + P)` | **mặc định** cho bài này |
| 4. Trung bình chạy | `avg += (d − avg)/cnt` | `O(1)` | `O(K + P)` | ngôn ngữ có tràn số, luồng dữ liệu vô hạn |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
Chuỗi lệnh: `checkIn(1,"A",2)`, `checkIn(2,"A",4)`, `checkOut(2,"B",9)`, `checkOut(1,"B",12)`, `getAverageTime("A","B")`. Kết quả truy vấn là bao nhiêu?

### Bài 2 (tính tay)
Viết nội dung `check_in` và `stats` sau từng lệnh của chuỗi: `checkIn(5,"X",1)`, `checkOut(5,"Y",7)`, `checkIn(5,"Y",8)`, `checkOut(5,"X",10)`, `checkIn(6,"X",11)`, `checkOut(6,"Y",13)`. Cho biết `getAverageTime("X","Y")` và `getAverageTime("Y","X")`.

### Bài 3 (điền TODO)
Hoàn thiện lớp `MyUnderground` (Cách 3) — các chỗ `TODO`.

### Bài 4 (tự viết lại)
Viết `UndergroundMinMax`: như `UndergroundSystem` và thêm `getMinTime(start, end)`, `getMaxTime(start, end)` chạy `O(1)`.

### Bài 5 (mini challenge)
Viết `UndergroundFrom`: thêm `getAverageFrom(start)` = thời gian trung bình của **mọi** chuyến xuất phát tại `start` (bất kể ga đến), `O(1)`.

In [26]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
class MyUnderground:
    def __init__(self):
        # TODO: tạo hai dict
        pass

    def checkIn(self, id, stationName, t):
        # TODO: lưu (stationName, t) theo id
        pass

    def checkOut(self, id, stationName, t):
        # TODO: pop thông tin vào, cộng thời gian vào stats[(start, stationName)]
        pass

    def getAverageTime(self, startStation, endStation):
        # TODO: trả tổng / số chuyến (chia số thực)
        pass

# Bài 4 — TODO
class UndergroundMinMax:
    # TODO: lưu thêm min, max cho mỗi cặp ga
    pass

# Bài 5 — TODO
class UndergroundFrom:
    # TODO: thêm dict from_stats: start -> [tổng, số]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Khách 2: 9 − 4 = 5. (2) Khách 1: 12 − 2 = 10 (thứ tự check-out không quan trọng). (3) Trung bình `(5 + 10)/2 = 7.5`.

**Bài 2.** (1) Sau lệnh 2: `stats = {(X,Y): [6,1]}`, `check_in` rỗng. (2) Sau lệnh 4: thêm `(Y,X): [2,1]`. (3) Sau lệnh 6: `(X,Y): [8,2]` ⇒ `X→Y = 4.0`, `Y→X = 2.0`.

**Bài 3.** (1) `self.check_in = {}`, `self.stats = {}`. (2) `start, t1 = self.check_in.pop(id)`. (3) `setdefault(..., [0, 0])`, cộng tổng và đếm; trả `total / cnt`.

**Bài 4.** (1) Giá trị cặp ga: `[tổng, số, min, max]`. (2) Khởi tạo `min = +inf`, `max = -inf`. (3) Mỗi check-out cập nhật `min(...)`, `max(...)`.

**Bài 5.** (1) Kế thừa `UndergroundSystem`. (2) Trong `checkOut`, đọc `start, t1 = self.check_in[id]` **trước** khi gọi `super().checkOut(...)` (vì lớp cha sẽ `pop`), rồi cập nhật `from_stats[start]`. (3) Kiểm chứng với baseline lưu mọi chuyến.

## 19. Đáp án tham khảo

In [27]:
# Bài 1, 2
o = run(UndergroundSystem,
        ["UndergroundSystem", "checkIn", "checkIn", "checkOut", "checkOut", "getAverageTime"],
        [[], [1, "A", 2], [2, "A", 4], [2, "B", 9], [1, "B", 12], ["A", "B"]])
assert abs(o[-1] - 7.5) <= 1e-5
u = UndergroundSystem()
for op, a in [("checkIn", (5, "X", 1)), ("checkOut", (5, "Y", 7)), ("checkIn", (5, "Y", 8)),
              ("checkOut", (5, "X", 10)), ("checkIn", (6, "X", 11)), ("checkOut", (6, "Y", 13))]:
    getattr(u, op)(*a)
assert u.stats == {("X", "Y"): [8, 2], ("Y", "X"): [2, 1]} and u.check_in == {}
assert u.getAverageTime("X", "Y") == 4.0 and u.getAverageTime("Y", "X") == 2.0
print("Bài 1, 2: đúng")

Bài 1, 2: đúng


In [28]:
# Bài 3 — đáp án
class MyUndergroundSol:
    def __init__(self):
        self.check_in = {}
        self.stats = {}

    def checkIn(self, id, stationName, t):
        self.check_in[id] = (stationName, t)

    def checkOut(self, id, stationName, t):
        start, t1 = self.check_in.pop(id)
        s = self.stats.setdefault((start, stationName), [0, 0])
        s[0] += t - t1
        s[1] += 1

    def getAverageTime(self, startStation, endStation):
        total, cnt = self.stats[(startStation, endStation)]
        return total / cnt

for ops, args, want in cases:
    assert same(run(MyUndergroundSol, ops, args), want)
print("Bài 3: đúng")

Bài 3: đúng


In [29]:
# Bài 4 — đáp án
class UndergroundMinMaxSol(UndergroundSystem):
    def __init__(self):
        super().__init__()
        self.extreme = {}                  # (start, end) -> [min, max]

    def checkOut(self, id, stationName, t):
        start, t1 = self.check_in[id]      # đọc trước khi lớp cha pop
        super().checkOut(id, stationName, t)
        d = t - t1
        e = self.extreme.setdefault((start, stationName), [math.inf, -math.inf])
        e[0] = min(e[0], d); e[1] = max(e[1], d)

    def getMinTime(self, startStation, endStation):
        return self.extreme[(startStation, endStation)][0]

    def getMaxTime(self, startStation, endStation):
        return self.extreme[(startStation, endStation)][1]

rng = random.Random(4)
for _ in range(200):
    ops, args = random_ops(rng, rng.randint(1, 50))
    ref = UndergroundBrute(); obj = UndergroundMinMaxSol()
    for op, a in zip(ops[1:], args[1:]):
        getattr(ref, op)(*a); getattr(obj, op)(*a)
    for (s, e) in obj.extreme:
        ds = [d for x, y, d in ref.trips if (x, y) == (s, e)]
        assert obj.getMinTime(s, e) == min(ds) and obj.getMaxTime(s, e) == max(ds)
        assert abs(obj.getAverageTime(s, e) - sum(ds) / len(ds)) <= 1e-5
print("Bài 4: đúng")

Bài 4: đúng


In [30]:
# Bài 5 — đáp án
class UndergroundFromSol(UndergroundSystem):
    def __init__(self):
        super().__init__()
        self.from_stats = {}               # start -> [tổng, số]

    def checkOut(self, id, stationName, t):
        start, t1 = self.check_in[id]
        super().checkOut(id, stationName, t)
        f = self.from_stats.setdefault(start, [0, 0])
        f[0] += t - t1; f[1] += 1

    def getAverageFrom(self, startStation):
        total, cnt = self.from_stats[startStation]
        return total / cnt

rng = random.Random(5)
for _ in range(200):
    ops, args = random_ops(rng, rng.randint(1, 50))
    ref = UndergroundBrute(); obj = UndergroundFromSol()
    for op, a in zip(ops[1:], args[1:]):
        getattr(ref, op)(*a); getattr(obj, op)(*a)
    for s in obj.from_stats:
        ds = [d for x, _, d in ref.trips if x == s]
        assert abs(obj.getAverageFrom(s) - sum(ds) / len(ds)) <= 1e-5
print("Bài 5: đúng")

Bài 5: đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "ghép sự kiện mở/đóng + thống kê theo nhóm":**

```python
class System:
    def __init__(self):
        self.open = {}                       # id -> (nơi bắt đầu, t bắt đầu)
        self.stats = {}                      # (bắt đầu, kết thúc) -> [tổng, số]

    def start(self, id, place, t):
        self.open[id] = (place, t)

    def end(self, id, place, t):
        p0, t0 = self.open.pop(id)
        s = self.stats.setdefault((p0, place), [0, 0])
        s[0] += t - t0; s[1] += 1

    def average(self, a, b):
        total, cnt = self.stats[(a, b)]
        return total / cnt                   # chia số thực
```

**Checklist trước khi submit:**
- [ ] Check-in lưu theo **id khách**, không theo ga.
- [ ] Khoá thống kê là tuple **có thứ tự** `(start, end)`.
- [ ] Không ghép chuỗi trần làm khoá.
- [ ] `pop` khi check-out để giải phóng bộ nhớ.
- [ ] Trung bình dùng `/`, không dùng `//`.
- [ ] Khi tự test: so số thực với sai số `1e-5`.
- [ ] Đúng tên lớp/phương thức: `UndergroundSystem`, `checkIn`, `checkOut`, `getAverageTime`.